In [46]:
import pandas as pd
import numpy as np

In [47]:
import warnings
warnings.filterwarnings('ignore')

In [48]:
df = pd.read_csv("data/household_power_consumption.txt", sep=';', low_memory=False)

In [49]:
df['datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], errors='coerce')

In [50]:
df = df.set_index('datetime')
df = df.apply(pd.to_numeric, errors='coerce')
df = df.fillna(method='ffill').fillna(method='bfill')

In [51]:
df = df[['Global_active_power']]

In [52]:
df_hourly = df.resample('H').mean()

In [53]:
df_hourly['hour'] = df_hourly.index.hour
df_hourly['day'] = df_hourly.index.day

In [54]:
df_hourly['lag_1'] = df_hourly['Global_active_power'].shift(1)
df_hourly['lag_2'] = df_hourly['Global_active_power'].shift(2)
df_hourly['rolling_mean'] = df_hourly['Global_active_power'].rolling(3).mean()
df_hourly = df_hourly.dropna()
print(df_hourly.shape)

(34587, 6)


In [55]:
X = df_hourly.drop('Global_active_power', axis=1)
y = df_hourly['Global_active_power']

In [56]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [57]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression

In [59]:
n_features = X_scaled.shape[1]
sfs = SequentialFeatureSelector(
    LinearRegression(), 
    n_features_to_select=min(3, n_features-1),
    direction='forward'
)
X_selected = sfs.fit_transform(X_scaled, y)

In [60]:
from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)

In [61]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error

ridge = Ridge()
lasso = Lasso()

ridge_mse, lasso_mse = [], []

for train, test in tscv.split(X_selected):
    X_train, X_test = X_selected[train], X_selected[test]
    y_train, y_test = y.iloc[train], y.iloc[test]

    ridge.fit(X_train, y_train)
    lasso.fit(X_train, y_train)

    ridge_mse.append(mean_squared_error(y_test, ridge.predict(X_test)))
    lasso_mse.append(mean_squared_error(y_test, lasso.predict(X_test)))

print("Ridge:", np.mean(ridge_mse))
print("Lasso:", np.mean(lasso_mse))

Ridge: 1.2309203025319046e-06
Lasso: 0.7632436015635208


In [62]:
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

pcr = Pipeline([
    ('pca', PCA(n_components=5)),
    ('reg', Ridge())
])

In [70]:
from sklearn.cross_decomposition import PLSRegression
pls = PLSRegression(n_components=5)

In [66]:
pcr_mse = []
pls_mse = []

In [67]:
for train_idx, test_idx in tscv.split(X_scaled):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    pcr.fit(X_train, y_train)
    y_pred = pcr.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    pcr_mse.append(mse)

print("PCR:", np.mean(pcr_mse))

PCR: 1.2428620916493924e-06


In [71]:
for train_idx, test_idx in tscv.split(X_scaled):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    pls.fit(X_train, y_train)

    y_pred = pls.predict(X_test)
    y_pred = y_pred.ravel()  

    mse = mean_squared_error(y_test, y_pred)
    pls_mse.append(mse)

print("PLS:", np.mean(pls_mse))

PLS: 6.526019717116337e-29


In [72]:
results = {
    "Ridge": np.mean(ridge_mse),
    "Lasso": np.mean(lasso_mse),
    "PCR": np.mean(pcr_mse),
    "PLS": np.mean(pls_mse)
}

In [74]:
print("Final Comparison:")
for model, mse in results.items():
    print(f"{model}: {mse}")

Final Comparison:
Ridge: 1.2309203025319046e-06
Lasso: 0.7632436015635208
PCR: 1.2428620916493924e-06
PLS: 6.526019717116337e-29


In [75]:
best_model_name = "Ridge"

In [76]:
from sklearn.linear_model import Ridge

final_model = Ridge()
final_model.fit(X_selected, y)

Ridge()

In [78]:
import joblib
joblib.dump(final_model, "model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(sfs, "selector.pkl")
print("model.pkl saved successfully!")

model.pkl saved successfully!
